# Первичный аудит файлов и основного датасета

Цель – проверить локальные файлы и качество `go_arounds_augmented.csv.gz` перед постановкой табличного baseline. Остальные файлы инвентаризируем; их содержимое требует отдельного аудита. Здесь не обучаем модели и не заявляем ранний прогноз или оценку безопасности захода.


In [1]:
# Проверяем окружение, в котором выполняется ноутбук
# Версии и путь к ядру помогут воспроизвести аудит на другом компьютере
import sys
from pathlib import Path

import pandas as pd

print('Python:', sys.version)
print('Исполняемый файл ядра:', Path(sys.executable).name)
print('Рабочая папка:', Path.cwd().name)
print('pandas:', pd.__version__)


Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Исполняемый файл ядра: python.exe
Рабочая папка: Approach Risk
pandas: 3.0.6


In [2]:
# Собираем список исходных файлов перед выбором датасета для аудита
search_root = Path.cwd()
if search_root.name == 'notebooks':
    search_root = search_root.parent

expected_names = {
    'go_arounds_augmented.csv.gz',
    'go_arounds_minimal.csv.gz',
    'go_arounds_agg.csv.gz',
    'validation_table.xlsx',
    'validation_sample.zip',
    'osn23_landings_merged.parquet.gzip',
}

# Ищем по именам во вложенных папках: данные могут лежать в разных каталогах
found_files = [
    path for path in search_root.rglob('*')
    if path.is_file() and path.name in expected_names
]

# Сохраняем полный путь и размер каждого файла. MiB = 1024**2 байт
inventory = pd.DataFrame(
    [
        {
            'file': path.name,
            'path': str(path),
            'bytes': path.stat().st_size,
            'MiB': round(path.stat().st_size / 1024**2, 2),
        }
        for path in found_files
    ],
    columns=['file', 'path', 'bytes', 'MiB'],
)

display(inventory)
print(
    'Не найдены в этой папке:',
    sorted(expected_names - {path.name for path in found_files}),
)


                                 file                                                                                                          path       bytes      MiB
0               go_arounds_agg.csv.gz               .\data\raw\go_arounds_agg.csv.gz       15104     0.01
1         go_arounds_augmented.csv.gz         .\data\raw\go_arounds_augmented.csv.gz   172668264   164.67
2           go_arounds_minimal.csv.gz           .\data\raw\go_arounds_minimal.csv.gz    92836941    88.54
3  osn23_landings_merged.parquet.gzip  .\data\raw\osn23_landings_merged.parquet.gzip   331328945   315.98
4               validation_sample.zip               .\data\raw\validation_sample.zip  1436435646  1369.89
5               validation_table.xlsx               .\data\raw\validation_table.xlsx       82585     0.08

Не найдены в этой папке: []


In [3]:
# Начинаем с расширенной таблицы: в ней есть метка исхода и контекстные поля
candidates = [
    path for path in found_files
    if path.name == 'go_arounds_augmented.csv.gz'
]

# При нескольких копиях файла останавливаемся, чтобы не выбрать одну случайно
if len(candidates) != 1:
    raise ValueError(
        f'Найдено файлов: {len(candidates)}. Укажите data_path вручную.'
    )

# Первых 10 000 строк достаточно для знакомства со схемой
# Это не случайная выборка; выводы по ней не переносим на весь датасет
data_path = candidates[0]
sample = pd.read_csv(data_path, compression='gzip', nrows=10_000)

print('Размер прочитанной части (строки, столбцы):', sample.shape)
print('Столбцы:', sample.columns.tolist())

# Проверяем типы, заполненность полей и примеры записей
sample.info()
display(sample.head())


                        time  icao24 callsign airport runway  has_ga  n_approaches  n_rwy_approached registration typecode icaoaircrafttype  wtc  glide_slope_angle  has_intersection   rwy_length airport_country airport_region operator_country operator_region  wind_speed_knts  wind_dir_deg  wind_gust_knts  visibility_m  temperature_deg  press_sea_level_p      press_p weather_intensity weather_precipitation weather_desc weather_obscuration weather_other
0  2019-01-01 00:01:13+00:00  a3b110    AAY56    KAUS    18R   False             1                 1       N337NV     A319              L2J    M                3.0             False  3736.957464             USA  North_America              USA   North_America              0.0           0.0             NaN      16093.44             10.0             1015.2  1014.901083                []                    []           []                  []            []
1  2019-01-01 00:12:58+00:00  accee2  ASA1144    KAUS    18R   False             1      

Размер прочитанной части (строки, столбцы): (10000, 31)
Столбцы: ['time', 'icao24', 'callsign', 'airport', 'runway', 'has_ga', 'n_approaches', 'n_rwy_approached', 'registration', 'typecode', 'icaoaircrafttype', 'wtc', 'glide_slope_angle', 'has_intersection', 'rwy_length', 'airport_country', 'airport_region', 'operator_country', 'operator_region', 'wind_speed_knts', 'wind_dir_deg', 'wind_gust_knts', 'visibility_m', 'temperature_deg', 'press_sea_level_p', 'press_p', 'weather_intensity', 'weather_precipitation', 'weather_desc', 'weather_obscuration', 'weather_other']
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   time                   10000 non-null  str    
 1   icao24                 10000 non-null  str    
 2   callsign               10000 non-null  str    
 3   airport                10000 non-null  str    
 4   runway         

In [4]:
# Считаем пропуски, которые pandas распознал при чтении CSV
# Строки вроде '[]' не являются NaN: их содержимое проверим отдельно
missing = pd.DataFrame({
    'dtype': sample.dtypes.astype(str),
    'missing_count': sample.isna().sum(),
    'missing_percent': sample.isna().mean().mul(100).round(2),
})

display(missing.sort_values('missing_count', ascending=False))

# has_ga обозначает наличие ухода на второй круг, а не степень опасности захода
# Распределение ниже относится только к прочитанным 10 000 строкам
if 'has_ga' in sample.columns:
    display(
        sample['has_ga']
        .value_counts(dropna=False)
        .rename_axis('has_ga')
        .to_frame('count')
    )
else:
    print('Столбец has_ga отсутствует. Нужно проверить структуру CSV.')


                         dtype  missing_count  missing_percent
wind_gust_knts         float64           8599            85.99
operator_region            str           1402            14.02
operator_country           str           1402            14.02
icaoaircrafttype           str           1309            13.09
wtc                        str            934             9.34
press_sea_level_p      float64            930             9.30
typecode                   str            923             9.23
wind_dir_deg           float64            219             2.19
registration               str             56             0.56
visibility_m           float64             20             0.20
wind_speed_knts        float64             11             0.11
n_approaches             int64              0             0.00
time                       str              0             0.00
icao24                     str              0             0.00
callsign                   str              0          

        count
has_ga       
False    9938
True       62

## Проверка полного CSV

Первый просмотр ограничивался 10 000 строками. Далее используем все строки, сохраняя исходные значения без заполнения пропусков.


In [5]:
# Загружаем полный набор: статистика первых строк может отличаться от общей
df = pd.read_csv(data_path, compression='gzip', low_memory=False)
print('Полный размер:', df.shape)
print('Память таблицы, MiB:', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

# Разбираем время в UTC. Ошибки преобразования считаем отдельно
times = pd.to_datetime(df['time'], errors='coerce', utc=True)
print('Нераспознанные даты:', times.isna().sum())
print('Период:', times.min(), '–', times.max())
print('Аэропортов:', df['airport'].nunique(dropna=True))
print('Уникальных бортов:', df['icao24'].nunique(dropna=True))
display(df['has_ga'].value_counts(dropna=False).to_frame('count'))
print('Доля has_ga=True, %:', round(df['has_ga'].eq(True).mean() * 100, 4))


          count
has_ga         
False   8935481
True      33664

Полный размер: (8969145, 31)
Память таблицы, MiB: 9348.54
Нераспознанные даты: 0
Период: 2019-01-01 00:01:13+00:00 – 2019-12-30 23:59:59+00:00
Аэропортов: 176
Уникальных бортов: 54530
Доля has_ga=True, %: 0.3753


In [6]:
# Различаем пропуски pandas и строковые маркеры отсутствующего содержимого
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean().mul(100).round(2),
    'unique_count': df.nunique(dropna=True),
})
display(quality.sort_values('missing_percent', ascending=False))

text_columns = df.select_dtypes(include=['str', 'object', 'string']).columns
markers = pd.DataFrame({
    'empty_string': {col: df[col].eq('').sum() for col in text_columns},
    'empty_list_string': {col: df[col].eq('[]').sum() for col in text_columns},
})
display(markers.loc[markers.sum(axis=1).gt(0)])

# Полный дубль и совпадение предполагаемого ключа – разные проверки
# Ключ не объявляем окончательным без подтверждения единицы наблюдения
key = ['time', 'icao24', 'airport']
print('Лишних полных дублей:', df.duplicated().sum())
print('Лишних совпадений ключа', key, ':', df.duplicated(key).sum())
print('Строк с пропуском в ключе:', df[key].isna().any(axis=1).sum())


                         dtype  missing_count  missing_percent  unique_count
wind_gust_knts         float64        8265105            92.15            69
press_sea_level_p      float64        5296835            59.06           729
icaoaircrafttype           str        1074405            11.98            37
wtc                        str         831505             9.27             4
typecode                   str         827657             9.23           529
wind_dir_deg           float64         538920             6.01            52
glide_slope_angle      float64         435061             4.85            18
rwy_length             float64         435061             4.85           342
has_intersection        object         435061             4.85             2
registration               str         433028             4.83         50143
operator_region            str         353480             3.94             6
operator_country           str         353480             3.94           116

                       empty_string  empty_list_string
weather_intensity                 0            7794996
weather_precipitation             0            7794996
weather_desc                      0            7794996
weather_obscuration               0            7794996
weather_other                     0            7794996

Лишних полных дублей: 0
Лишних совпадений ключа ['time', 'icao24', 'airport'] : 0
Строк с пропуском в ключе: 0


In [7]:
# Проверяем распределение исходов по аэропортам и месяцам
# Это помогает заметить неоднородность выборки до выбора разбиения
by_airport = df.groupby('airport').agg(
    rows=('has_ga', 'size'), go_arounds=('has_ga', 'sum'),
)
by_airport['ga_percent'] = (by_airport['go_arounds'] / by_airport['rows'] * 100).round(3)
display(by_airport.sort_values('rows', ascending=False))

by_month = df.assign(month=times.dt.strftime('%Y-%m')).groupby('month').agg(
    rows=('has_ga', 'size'), go_arounds=('has_ga', 'sum'),
)
by_month['ga_percent'] = (by_month['go_arounds'] / by_month['rows'] * 100).round(3)
display(by_month)


           rows  go_arounds  ga_percent
airport                                
KORD     317521        1367       0.431
KDFW     290627         960       0.330
EGLL     226467         864       0.382
EDDF     225808         683       0.302
LFPG     215467         758       0.352
EHAM     201895         362       0.179
KSFO     197356        1021       0.517
LEMD     182049         295       0.162
KJFK     176759         583       0.330
OMDB     175634         515       0.293
EDDM     174630         535       0.306
KPHX     167126         455       0.272
YSSY     162284         480       0.296
KPHL     151025         829       0.549
KMSP     149493         582       0.389
VHHH     144010         323       0.224
KLAS     143830         314       0.218
KSEA     136372         583       0.428
UUEE     135292         574       0.424
KEWR     129941         508       0.391
VTBS     124360         150       0.121
KLGA     123199         816       0.662
KBOS     121888         399       0.327


           rows  go_arounds  ga_percent
month                                  
2019-01  634696        2577       0.406
2019-02  616874        2733       0.443
2019-03  656990        2696       0.410
2019-04  673813        2434       0.361
2019-05  717947        2578       0.359
2019-06  727833        2719       0.374
2019-07  807313        2770       0.343
2019-08  829358        2945       0.355
2019-09  772287        2641       0.342
2019-10  876435        3338       0.381
2019-11  818428        3025       0.370
2019-12  837171        3208       0.383

In [8]:
# Диапазоны помогают найти подозрительные значения; их не исправляем автоматически
numeric = df.select_dtypes(include='number')
display(numeric.describe(percentiles=[0.01, 0.5, 0.99]).T)

# Это базовые проверки единиц и диапазонов, а не полная авиационная валидация
checks = {
    'wind_speed_knts < 0': df['wind_speed_knts'].lt(0).sum(),
    'wind_gust_knts < 0': df['wind_gust_knts'].lt(0).sum(),
    'wind_dir_deg вне [0, 360]': (df['wind_dir_deg'].lt(0) | df['wind_dir_deg'].gt(360)).sum(),
    'visibility_m < 0': df['visibility_m'].lt(0).sum(),
    'rwy_length <= 0': df['rwy_length'].le(0).sum(),
    'n_approaches < 1': df['n_approaches'].lt(1).sum(),
    'n_rwy_approached < 1': df['n_rwy_approached'].lt(1).sum(),
}
display(pd.Series(checks, name='count').to_frame())

# Число попыток и ВПП может описывать уже состоявшийся исход
# Проверяем связь с меткой, но не включаем эти поля в прогнозные признаки
display(pd.crosstab(df['n_approaches'], df['has_ga']))
display(pd.crosstab(df['n_rwy_approached'], df['has_ga']))


                       count          mean          std          min           1%           50%           99%           max
n_approaches       8969145.0      1.004741     0.113284     1.000000     1.000000      1.000000      1.000000     31.000000
n_rwy_approached   8969145.0      1.000708     0.027200     1.000000     1.000000      1.000000      1.000000      6.000000
glide_slope_angle  8534084.0      3.018132     0.149180     2.500000     2.850000      3.000000      3.500000      5.500000
rwy_length         8534084.0   3203.329003   570.397711  1064.720999  1743.309405   3281.170466   4206.432463   4875.435865
wind_speed_knts    8962299.0      7.973403     4.750365     0.000000     0.000000      7.000000     22.000000    622.028800
wind_dir_deg       8430225.0    184.415292   103.449099     0.000000     0.000000    200.000000    360.000000    360.000000
wind_gust_knts      704040.0     23.296424     5.807931     9.719200    14.000000     22.000000     41.000000     87.000000
visibili

                           count
wind_speed_knts < 0            0
wind_gust_knts < 0             0
wind_dir_deg вне [0, 360]      0
visibility_m < 0               0
rwy_length <= 0                0
n_approaches < 1               0
n_rwy_approached < 1           0

has_ga          False  True 
n_approaches                
1             8935481      0
2                   0  30728
3                   0   1533
4                   0    372
5                   0    281
6                   0    177
7                   0    123
8                   0    123
9                   0     79
10                  0     47
11                  0     28
12                  0     38
13                  0     16
14                  0     14
15                  0     12
16                  0     13
17                  0     20
18                  0     19
19                  0      8
20                  0      6
21                  0      6
22                  0      7
23                  0      3
24                  0      4
25                  0      2
26                  0      1
28                  0      1
29                  0      1
31                  0      2

has_ga              False  True 
n_rwy_approached                
1                 8935481  27438
2                       0   6118
3                       0     96
4                       0      9
5                       0      2
6                       0      1

## Вывод по первичному аудиту

Все шесть ожидаемых файлов найдены. Полностью проверен основной CSV: **8,969,145 строк и 31 столбцов**, 176 аэропортов. Период записей: **2019-01-01 00:01:13+00:00 – 2019-12-30 23:59:59+00:00**. Нераспознанных дат: 0.

Метка `has_ga=True` встречается в **33,664 строках (0.375%)**. Классы несбалансированы; одной accuracy будет недостаточно для оценки baseline. Распределение по аэропортам и месяцам показано выше, его нужно учитывать при выборе разбиения.

Лишних полных дублей: **0**; лишних совпадений предполагаемого ключа `time + icao24 + airport`: **0**. Совпадение ключа само по себе не разрешает удаление строки: сначала нужно подтвердить единицу наблюдения.

В `wind_gust_knts` отсутствует **92.15%** значений. Отсутствие записи о порывах не приравниваем автоматически к нулевому ветру. Строки `[]` в погодных полях отделяем от NaN; их смысл уточняем по описанию источника.

Ненулевые результаты базовых проверок диапазонов: **не обнаружены**. Это проверка очевидных ограничений, а не подтверждение корректности всех физических величин и разметки.

### Ограничения и следующие решения

- `has_ga` – наблюдаемый исход ухода на второй круг; по нему нельзя установить опасность захода или причину маневра.
- `n_approaches` и `n_rwy_approached` исключаем из первого прогнозного набора как потенциальные признаки утечки. Связь с исходом показана в таблицах выше.
- Время доступности погоды, выбор ВПП и смысл `time` еще не подтверждены по первоисточнику. Пока допустима только ограниченная ретроспективная постановка.
- Файлы minimal, agg, parquet и материалы валидации здесь только инвентаризированы. Качество их содержимого и точность исходной разметки не проверены.
- Следующий шаг: словарь полей с происхождением и доступностью, затем фиксированное разбиение и Dummy baseline. Преобразования и заполнение пропусков обучаем только на train.

Первичный технический аудит основного CSV завершен. Модели не обучались; готовность данных для раннего прогнозирования этим аудитом не доказана.
